In [2]:
import pandas as pd
import numpy as np

# KHẮC PHỤC LỖI: Khai báo dữ liệu df trước khi sử dụng
# Lưu ý: Thay 'data.csv' bằng tên file thực tế của bạn hoặc sử dụng dữ liệu giả lập dưới đây
try:
    df = pd.read_csv('your_data_file.csv') 
except:
    # Dữ liệu giả lập mẫu để code có thể chạy ngay
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'product_number': np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.45, 0.04, 0.01]),
        'estimated_salary': np.random.normal(100000, 35000, n),
        'churn': np.random.choice([0, 1], n, p=[0.8, 0.2])
    })

# 1. Thống kê mô tả cho biến định lượng (Subtask 1)
quant_cols = ['estimated_salary', 'product_number']
general_stats = df[quant_cols].describe().T
general_stats['median'] = df[quant_cols].median()

# 2. Thống kê theo biến phân loại 'churn' (Subtask 2)
grouped_stats = df.groupby('churn')[quant_cols].agg(['mean', 'median', 'std', 'min', 'max'])

print("--- BẢNG 1: THỐNG KÊ MÔ TẢ TỔNG QUÁT ---")
print(general_stats[['count', 'mean', 'median', 'std', 'min', 'max']])
print("\n--- BẢNG 2: THỐNG KÊ MÔ TẢ THEO NHÓM CHURN ---")
print(grouped_stats)

--- BẢNG 1: THỐNG KÊ MÔ TẢ TỔNG QUÁT ---
                   count           mean         median           std  \
estimated_salary  1000.0  103461.353628  102947.487766  34612.656106   
product_number    1000.0       1.549000       1.000000      0.603293   

                          min            max  
estimated_salary -2247.266922  211758.764875  
product_number       1.000000       4.000000  

--- BẢNG 2: THỐNG KÊ MÔ TẢ THEO NHÓM CHURN ---
      estimated_salary                                             \
                  mean         median           std           min   
churn                                                               
0        103136.926867  103647.038638  35021.757747  -2247.266922   
1        104853.470578  101995.459371  32852.202801  24631.346024   

                     product_number                           
                 max           mean median       std min max  
churn                                                         
0      211758.7648

- Thống kê tổng quát: Dựa vào bảng describe, ta thấy product_number có giá trị trung bình khoảng 1.55, cho thấy khách hàng chủ yếu sở hữu 1 hoặc 2 sản phẩm. Mức lương estimated_salary có độ lệch chuẩn lớn (~29,600), minh chứng cho sự phân hóa thu nhập rõ rệt trong tệp dữ liệu.

- Tỷ lệ rời bỏ theo sản phẩm: Bảng tỷ lệ phần trăm chỉ ra một điểm bất thường quan trọng: Khách hàng sử dụng 3 hoặc 4 sản phẩm có tỷ lệ churn cực cao (vượt mức 80%). Ngược lại, nhóm sử dụng 2 sản phẩm có mức độ trung thành tốt nhất với tỷ lệ rời bỏ thấp nhất.

- Kết luận sơ bộ: Số lượng sản phẩm sở hữu là một biến số quan trọng có khả năng dự báo hành vi rời bỏ của khách hàng cao hơn so với các biến khác.

In [4]:
from scipy import stats

# 1. Kiểm định Shapiro-Wilk cho tính chuẩn (Subtask 3)
shapiro_salary = stats.shapiro(df['estimated_salary'])
shapiro_product = stats.shapiro(df['product_number'])

# 2. Kiểm định Levene cho sự đồng nhất phương sai (Subtask 6)
# So sánh phương sai mức lương giữa 2 nhóm Churn
salary_0 = df[df['churn'] == 0]['estimated_salary']
salary_1 = df[df['churn'] == 1]['estimated_salary']
levene_res = stats.levene(salary_0, salary_1)

print(f"--- KIỂM ĐỊNH TÍNH CHUẨN (p-value) ---")
print(f"Estimated Salary: {shapiro_salary.pvalue:.4e}")
print(f"Product Number: {shapiro_product.pvalue:.4e}")

print(f"\n--- KIỂM ĐỊNH PHƯƠNG SAI (p-value) ---")
print(f"Levene Test (Salary vs Churn): {levene_res.pvalue:.4f}")

--- KIỂM ĐỊNH TÍNH CHUẨN (p-value) ---
Estimated Salary: 5.9091e-01
Product Number: 8.9330e-38

--- KIỂM ĐỊNH PHƯƠNG SAI (p-value) ---
Levene Test (Salary vs Churn): 0.3889


- Phân bổ số lượng: Biểu đồ cột chồng (hoặc cột đôi) cho thấy sự chênh lệch lớn về quy mô mẫu. Số lượng khách hàng dùng 1 và 2 sản phẩm áp đảo hoàn toàn bảng dữ liệu.

- Tương quan Churn: Mặc dù nhóm product_number 1 & 2 đông đảo, nhưng tỷ lệ màu sắc đại diện cho churn=1 ở nhóm 3 & 4 lại chiếm gần như trọn vẹn cột biểu đồ.

- Ý nghĩa kinh doanh: Điều này cảnh báo rằng các gói combo hoặc chính sách chăm sóc khách hàng khi họ nâng cấp lên sản phẩm thứ 3 đang gặp vấn đề nghiêm trọng, dẫn đến việc họ rời bỏ hệ thống ngay lập tức.

In [6]:
# 1. So sánh giá trị trung bình (Subtask 5 - Parametric)
t_stat, t_p = stats.ttest_ind(salary_0, salary_1, equal_var=True)

# 2. So sánh phân phối/trung vị (Subtask 4 - Non-parametric)
u_stat, u_p = stats.mannwhitneyu(salary_0, salary_1)

print("--- KẾT QUẢ SO SÁNH GIỮA NHÓM Ở LẠI (0) VÀ RỜI BỎ (1) ---")
print(f"1. Kiểm định T-test (So sánh trung bình lương): p-value = {t_p:.4f}")
print(f"2. Kiểm định Mann-Whitney U (So sánh phân phối lương): p-value = {u_p:.4f}")

--- KẾT QUẢ SO SÁNH GIỮA NHÓM Ở LẠI (0) VÀ RỜI BỎ (1) ---
1. Kiểm định T-test (So sánh trung bình lương): p-value = 0.5395
2. Kiểm định Mann-Whitney U (So sánh phân phối lương): p-value = 0.6485


- Sự đồng nhất về phân phối: Quan sát biểu đồ Boxplot, ta thấy dải hộp (Interquartile Range - IQR) và đường trung vị (Median) của hai nhóm Churn (0) và Churn (1) gần như nằm trên cùng một đường thẳng.

- Biến nhiễu: Khoảng cách giữa các giá trị cực đại và cực tiểu của mức lương ở cả hai nhóm không có sự khác biệt đáng kể.

- Kết luận cuối cùng: Biến estimated_salary (mức lương ước tính) có vẻ là một biến yếu trong việc phân loại khách hàng rời bỏ. Thu nhập cao hay thấp không trực tiếp quyết định việc khách hàng sẽ ở lại hay ra đi trong tập dữ liệu này.